In [ ]:
# =========================
# 1. IMPORTACIONES Y CARGA DEL CSV
# =========================
import io
import pandas as pd

from google.colab import files

uploaded = files.upload()

# Coge automáticamente el nombre del archivo subido
file_name = list(uploaded.keys())[0]

# Lee el CSV
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

# Ver primeras filas
df.head()

Saving Bank Customer Churn Prediction.csv to Bank Customer Churn Prediction (2).csv


,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
print(df.columns.tolist())

['customer_id', 'credit_score', 'country', 'gender', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'estimated_salary', 'churn']


In [ ]:
# =========================
# 2. LIMPIEZA BÁSICA
# =========================
df = df.drop(['customer_id'], axis=1, errors='ignore')

# Variable objetivo
y = df['churn']

# Variables predictoras
X = df.drop('churn', axis=1)

print("Tamaño de X:", X.shape)
print("Tamaño de y:", y.shape)
print("\nColumnas de X:")
print(X.columns.tolist())

Tamaño de X: (10000, 10)
Tamaño de y: (10000,)

Columnas de X:
['credit_score', 'country', 'gender', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'estimated_salary']


In [ ]:
# =========================
# 3. IDENTIFICAR COLUMNAS
# =========================
categorical_cols = [col for col in X.columns if X[col].dtype == 'object']
numerical_cols = [col for col in X.columns if X[col].dtype in ['int64', 'float64']]

print("Columnas categóricas:", categorical_cols)
print("Columnas numéricas:", numerical_cols)

Columnas categóricas: ['country', 'gender']
Columnas numéricas: ['credit_score', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'estimated_salary']


In [ ]:
# =========================
# 4. PREPROCESADO
# =========================
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numerical_transformer = SimpleImputer(strategy='median')

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print("Preprocesado creado correctamente")

Preprocesado creado correctamente


In [ ]:
# =========================
# 5. RANDOM FOREST
# =========================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf_model)
])

rf_scores = cross_val_score(
    rf_pipeline,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print("Random Forest - Accuracy por fold:", rf_scores)
print("Random Forest - Accuracy media:", round(rf_scores.mean(), 4))

Random Forest - Accuracy por fold: [0.8565 0.871  0.862  0.8655 0.8585]
Random Forest - Accuracy media: 0.8627


In [ ]:
!pip install xgboost

In [ ]:
# =========================
# 6. XGBOOST
# =========================
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    eval_metric='logloss'
)

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

xgb_scores = cross_val_score(
    xgb_pipeline,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print("XGBoost - Accuracy por fold:", xgb_scores)
print("XGBoost - Accuracy media:", round(xgb_scores.mean(), 4))

XGBoost - Accuracy por fold: [0.8655 0.873  0.8585 0.8695 0.855 ]
XGBoost - Accuracy media: 0.8643


In [ ]:
# =========================
# 7. COMPARACIÓN FINAL
# =========================
print("Random Forest - Accuracy media:", round(rf_scores.mean(), 4))
print("XGBoost - Accuracy media:", round(xgb_scores.mean(), 4))

if xgb_scores.mean() > rf_scores.mean():
    print("Mejor modelo: XGBoost")
else:
    print("Mejor modelo: Random Forest")

Random Forest - Accuracy media: 0.8627
XGBoost - Accuracy media: 0.8643
Mejor modelo: XGBoost
